# Autofiltering Validation Dataset

In [48]:
import pandas as pd
import json

from pathlib import Path

In [49]:
fp = Path("../evaluation_data/validation.jsonl")
df = pd.DataFrame([json.loads(line) for line in open(fp, 'r', 
													encoding="utf-8")])
df.info()
df.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6433 entries, 0 to 6432
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   text       6433 non-null   object
 1   languages  6433 non-null   object
dtypes: object(2)
memory usage: 100.6+ KB


,text,languages
0,Det spesielle ved den reduktive fysikalismen e...,[nn]
1,"Han ledet en gruppe på 16, og det var ingen li...",[nb]
2,Հայաստանի Հանրապետության կառավարության որոշում,[other]
3,Jeg tror aldri han har hatt en uvenn,[nb]
4,Straks vi innser at denne sjølvavsløringa ligg...,[nn]


Remove `other`

In [50]:
filtered = df[df['languages'].apply(lambda x: 'other' in x)]
filtered.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1116 entries, 2 to 6429
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   text       1116 non-null   object
 1   languages  1116 non-null   object
dtypes: object(2)
memory usage: 26.2+ KB


In [51]:
df = df[~df['languages'].apply(lambda x: 'other' in x)]
df.info()
df.head()

<class 'pandas.core.frame.DataFrame'>
Index: 5317 entries, 0 to 6432
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   text       5317 non-null   object
 1   languages  5317 non-null   object
dtypes: object(2)
memory usage: 124.6+ KB


,text,languages
0,Det spesielle ved den reduktive fysikalismen e...,[nn]
1,"Han ledet en gruppe på 16, og det var ingen li...",[nb]
3,Jeg tror aldri han har hatt en uvenn,[nb]
4,Straks vi innser at denne sjølvavsløringa ligg...,[nn]
5,Det er gjerne det som gjer offentleg sektor ti...,[nn]


Define indicators per language and remove instances accordingly:

In [52]:
indicators = {
    'da': [
		"noget",
		"af",
		"endnu",
		"uden",
		"inde",
		"mellem",
		"ud",
		"undtagen",
		"gennem",
		"specielt",
		"blevet",
		"afslag",
		"sidst"
	], 
	'nb': [
		"hva",
	], 
	'nn': [
		"ikkje",
  		"mykje",
		"frå",
		"kva",
		"eit",
		"ein",
  		"sjølvsagt",
    	"sjølv",
		"gjekk",
		"noko",
		"tidleg",
		"gjer",
		"tidlegare",
		"kjem",
  		"meir",
		"berre",
		"desse",
		"merkeleg"
	], 
	'sv': [
		"jag",
		"vad",
		"säger",
		"också",
		"för",
		"och",
		"är",
		"är.",
		"själv",
		"görs",
		"mellan",
		"någon",
		"ifrån",
		"från",
	], 
}

indicies = []
for l, i in indicators.items():
    temp =  df[df['languages'].apply(lambda x: l in x)]
    temp = temp.assign(_text=[x.lower().split() for x in temp["text"]])
    
    for word in i:
        _temp = temp[temp['_text'].apply(lambda x: word in x)]
        indicies.extend(_temp.index)

indicies = list(set(indicies))
res = df.loc[indicies]
res.info()
res["languages"].value_counts()

<class 'pandas.core.frame.DataFrame'>
Index: 1525 entries, 0 to 6418
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   text       1525 non-null   object
 1   languages  1525 non-null   object
dtypes: object(2)
memory usage: 35.7+ KB


languages
[nn]    951
[sv]    358
[da]    167
[nb]     49
Name: count, dtype: int64

Add `res` to the `filtered` data frame and create `original` column:

In [53]:
filtered = pd.concat([filtered, res])
filtered["original"] = filtered["languages"]
filtered.info()

<class 'pandas.core.frame.DataFrame'>
Index: 2641 entries, 2 to 6418
Data columns (total 3 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   text       2641 non-null   object
 1   languages  2641 non-null   object
 2   original   2641 non-null   object
dtypes: object(3)
memory usage: 82.5+ KB


Shuffle and save the automatically filtered instances to file:

In [54]:
filtered = filtered.sample(frac=1, random_state=42).reset_index(drop=True)

filtered.to_json("validate_auto_filtered.jsonl", orient="records", lines=True)

## Splitting Remaining instances for manual annotation

In [56]:
remaining = df[~df.index.isin(indicies)]
remaining.info()
remaining["languages"].value_counts()

<class 'pandas.core.frame.DataFrame'>
Index: 3792 entries, 1 to 6432
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   text       3792 non-null   object
 1   languages  3792 non-null   object
dtypes: object(2)
memory usage: 88.9+ KB


languages
[nb]    2343
[nn]     908
[da]     396
[sv]     145
Name: count, dtype: int64

Filtering out `sv` and `da` for non-native annotater:

In [57]:
sv = remaining[remaining['languages'].apply(lambda x: "sv" in x)]
da = remaining[remaining['languages'].apply(lambda x: "da" in x)]

sv_da = pd.concat((sv, da))
sv_da.info()

<class 'pandas.core.frame.DataFrame'>
Index: 541 entries, 35 to 6420
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   text       541 non-null    object
 1   languages  541 non-null    object
dtypes: object(2)
memory usage: 12.7+ KB


In [ ]:
sv_da.to_json("validation_subset_da_sv.jsonl", orient="records", lines=True)

Split the remaining instances into for our 5 other annotaters:

In [58]:
remaining = remaining.drop(sv_da.index)
remaining.info()

<class 'pandas.core.frame.DataFrame'>
Index: 3251 entries, 1 to 6431
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   text       3251 non-null   object
 1   languages  3251 non-null   object
dtypes: object(2)
memory usage: 76.2+ KB


In [77]:
interval = len(remaining) // 5
template = "validation_subset_{i}.jsonl"


for i in range(4):
    temp = remaining.loc[i * interval: (i+1)*interval-1]
    temp.to_json(template.format(i=i), 
                 orient="records", 
                 lines=True)
temp = remaining.loc[(i+1)*interval-1:]
temp.to_json(template.format(i=i), 
                 orient="records", 
                 lines=True)
